# tcml-workflow quickstart

Minimal walkthrough of the pipeline from Phonix download → descriptors → Nested-CV training → candidate prediction. Designed to run in under 5 minutes on a laptop with a subset of 200 materials.

## 1. Download Phonix-summary (CC-BY-4.0)

Skips if `data/phonix/summary.parquet` already exists. Full download is 149 MB.

In [ ]:
!python ../scripts/01_download_phonix.py --out_dir ../data/phonix

## 2. Inspect the cleaned training pool

In [ ]:
from tcml_workflow import PhonixLoader

loader = PhonixLoader('../data/phonix/summary.parquet')
df = loader.clean()
print(loader.summary())
df[['mp_id', 'formula', 'spg_number', 'klat[W/mK]', 'log_k', 'fc3_error[%]']].head(10)

## 3. Compute descriptors (small subset for the demo)

In [ ]:
!python ../scripts/02_compute_descriptors.py \
    --phonix ../data/phonix/summary.parquet \
    --out_dir ../data/processed_demo \
    --max_materials 200

## 4. Train one algorithm (XGBoost) on the demo subset

In [ ]:
!python ../scripts/03_train.py \
    --algorithm XGBoost \
    --train ../data/processed_demo/train.parquet \
    --out_dir ../results_demo \
    --n_iter 20

## 5. Aggregate metrics

In [ ]:
!python ../scripts/04_evaluate.py --results_dir ../results_demo

## Next steps

- For folds in which no chemical formula appears on both sides of a split, run `scripts/27_database_structure_variables.py` and then `scripts/35_grouped_nested_cv.py` (several hours).
- Train SVR and GPR with the same command, swapping `--algorithm`.
- Run `scripts/05_predict_candidates.py` on your own structures (example: `data/candidates_example.csv`).